# AstroPaperRec: arXiv Astrophysics Paper Recommendation Prototype

This notebook demonstrates a feature prototype for **AstroPaperRec**, a content-based recommendation system for astrophysics research papers. The aim of the prototype is to recommend arXiv papers that are textually similar to a selected seed paper.

The prototype uses the arXiv metadata dataset and filters it to astrophysics-related papers. For each paper, the title and abstract are combined and transformed into TF-IDF vectors. Cosine similarity is then used to compare the seed paper with other papers in the dataset and return the top-ranked recommendations.

The prototype also calculates **Category Precision@K** as a proxy evaluation metric. A recommended paper is counted as relevant if it shares at least one arXiv category with the seed paper. Since arXiv categories can be broad, this metric is used together with manual inspection of the recommendation results.


In [1]:
import os
import json
import re

import kagglehub
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

In [2]:
path = kagglehub.dataset_download("Cornell-University/arxiv")

print("Path to dataset files:", path)
print(os.listdir(path))

DATA_PATH = os.path.join(path, "arxiv-metadata-oai-snapshot.json")
print(DATA_PATH)

Path to dataset files: /Users/cychang/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/305
['arxiv-metadata-oai-snapshot.json']
/Users/cychang/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/305/arxiv-metadata-oai-snapshot.json


In [3]:
def is_astrophysics_paper(categories: str) -> bool:
    """
    Return True if the paper belongs to any astro-ph category.
    """
    if not isinstance(categories, str):
        return False
    return any(cat.startswith("astro-ph") for cat in categories.split())


def clean_text(text: str) -> str:
    #Basic text cleaning for titles and abstracts.
    
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def load_astrophysics_subset(file_path: str, max_papers: int = 5000) -> pd.DataFrame:
    """
    Load arXiv metadata line by line and keep only astrophysics papers.
    """
    records = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            paper = json.loads(line)

            categories = paper.get("categories", "")

            if is_astrophysics_paper(categories):
                records.append({
                    "id": paper.get("id", ""),
                    "title": paper.get("title", ""),
                    "abstract": paper.get("abstract", ""),
                    "authors": paper.get("authors", ""),
                    "categories": categories,
                    "update_date": paper.get("update_date", "")
                })

            if len(records) >= max_papers:
                break

    df = pd.DataFrame(records)

    df = df.dropna(subset=["title", "abstract", "categories"])
    df = df.drop_duplicates(subset=["id"])

    df["text"] = (
        df["title"].fillna("") + " " + df["abstract"].fillna("")
    ).apply(clean_text)

    df = df[df["text"].str.len() > 0].reset_index(drop=True)

    return df

In [4]:
def audit_astrophysics_dataset(file_path: str):
    """
    Scan the complete arXiv metadata file and collect statistics
    required for Tables 3.1 and 3.2.

    This function does not apply the 5,000-paper prototype limit.
    """

    total_records = 0
    astro_records = 0
    valid_title_abstract = 0

    astro_data = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            total_records += 1

            paper = json.loads(line)

            categories = paper.get("categories", "")
            title = paper.get("title", "")
            abstract = paper.get("abstract", "")

            # Keep only astrophysics papers
            if not is_astrophysics_paper(categories):
                continue

            astro_records += 1

            # Check whether title and abstract are usable
            if (
                isinstance(title, str)
                and title.strip()
                and isinstance(abstract, str)
                and abstract.strip()
            ):
                valid_title_abstract += 1

            astro_data.append({
                "id": paper.get("id", ""),
                "title": title,
                "abstract": abstract,
                "categories": categories,
                "update_date": paper.get("update_date", "")
            })

    # Convert complete astrophysics subset to DataFrame
    astro_df = pd.DataFrame(astro_data)

    # Keep valid title + abstract records
    valid_df = astro_df[
        astro_df["title"].notna()
        & astro_df["abstract"].notna()
        & (astro_df["title"].str.strip() != "")
        & (astro_df["abstract"].str.strip() != "")
    ].copy()

    before_duplicates = len(valid_df)

    # Remove duplicate paper IDs
    valid_df = valid_df.drop_duplicates(subset=["id"])

    duplicates_removed = before_duplicates - len(valid_df)

    # Construct and clean document text
    valid_df["text"] = (
        valid_df["title"].fillna("")
        + " "
        + valid_df["abstract"].fillna("")
    ).apply(clean_text)

    before_empty_text = len(valid_df)

    valid_df = valid_df[
        valid_df["text"].str.len() > 0
    ].copy()

    empty_text_removed = before_empty_text - len(valid_df)

    valid_df = valid_df.reset_index(drop=True)

    # --------------------------------------------------
    # Table 3.2 statistics
    # --------------------------------------------------

    abstract_lengths = (
        valid_df["abstract"]
        .str.split()
        .str.len()
    )

    astro_categories = set()

    for categories in valid_df["categories"]:
        for category in categories.split():
            if category.startswith("astro-ph"):
                astro_categories.add(category)

    dates = pd.to_datetime(
        valid_df["update_date"],
        errors="coerce"
    )

    # --------------------------------------------------
    # Print report-ready statistics
    # --------------------------------------------------

    print("=" * 60)
    print("TABLE 3.1 — DATASET SELECTION")
    print("=" * 60)

    print(f"Complete metadata dataset:        {total_records:,}")
    print(f"Papers containing astro-ph:       {astro_records:,}")
    print(f"Valid title + abstract:           {valid_title_abstract:,}")
    print(f"Duplicates removed:               {duplicates_removed:,}")
    print(f"Empty processed text removed:     {empty_text_removed:,}")
    print(f"Final astrophysics dataset:       {len(valid_df):,}")

    print()
    print("=" * 60)
    print("TABLE 3.2 — DATASET CHARACTERISTICS")
    print("=" * 60)

    print(f"Number of papers:                 {len(valid_df):,}")
    print(f"Mean abstract length:             {abstract_lengths.mean():.2f} words")
    print(f"Median abstract length:           {abstract_lengths.median():.0f} words")
    print(f"Minimum abstract length:          {abstract_lengths.min():,} words")
    print(f"Maximum abstract length:          {abstract_lengths.max():,} words")
    print(f"Number of astro-ph categories:    {len(astro_categories)}")

    if dates.notna().any():
        print(
            f"Publication/update date range:     "
            f"{dates.min().date()} – {dates.max().date()}"
        )

    print()
    print("Astrophysics categories:")
    for category in sorted(astro_categories):
        print(f"  - {category}")

    return valid_df

In [5]:
full_astro_df = audit_astrophysics_dataset(DATA_PATH)

TABLE 3.1 — DATASET SELECTION
Complete metadata dataset:        3,182,867
Papers containing astro-ph:       393,390
Valid title + abstract:           393,390
Duplicates removed:               0
Empty processed text removed:     0
Final astrophysics dataset:       393,390

TABLE 3.2 — DATASET CHARACTERISTICS
Number of papers:                 393,390
Mean abstract length:             187.91 words
Median abstract length:           193 words
Minimum abstract length:          1 words
Maximum abstract length:          584 words
Number of astro-ph categories:    7
Publication/update date range:     2007-05-23 – 2026-09-25

Astrophysics categories:
  - astro-ph
  - astro-ph.CO
  - astro-ph.EP
  - astro-ph.GA
  - astro-ph.HE
  - astro-ph.IM
  - astro-ph.SR


In [6]:
model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [39]:
MAX_ASTRO_PAPERS = len(full_astro_df)
TOP_K = 10

In [ ]:
df = load_astrophysics_subset(DATA_PATH, max_papers=MAX_ASTRO_PAPERS)

print(f"Loaded {len(df)} astrophysics papers.")
df.head()

In [ ]:
sbert_texts = (
    df["title"].fillna("").str.strip()
    + ". "
    + df["abstract"].fillna("").str.strip()
).tolist()

In [ ]:
sbert_embeddings = model.encode(
    sbert_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("SBERT model:", model_name)
print("Number of documents:", len(sbert_texts))
print("Embedding dimension:", sbert_embeddings.shape[1])
print("Embedding matrix shape:", sbert_embeddings.shape)

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    min_df=2,
    max_df=0.85
)

tfidf_matrix = vectorizer.fit_transform(df["text"])

print("Number of documents:", tfidf_matrix.shape[0])
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("TF-IDF matrix shape:", tfidf_matrix.shape)

print("\nParameters:")
print("stop_words =", vectorizer.stop_words)
print("max_features =", vectorizer.max_features)
print("ngram_range =", vectorizer.ngram_range)
print("min_df =", vectorizer.min_df)
print("max_df =", vectorizer.max_df)

In [ ]:
def search_seed_paper(df: pd.DataFrame, keyword: str, max_results: int = 10) -> pd.DataFrame:
    """
    Search for possible seed papers by keyword in the title.
    """
    keyword = keyword.lower()

    matches = df[df["title"].str.lower().str.contains(keyword, na=False)]

    return matches[[
        "id",
        "title",
        "authors",
        "categories",
        "update_date"
    ]].head(max_results)


def recommend_similar_papers(seed_index: int, df: pd.DataFrame, tfidf_matrix, k: int = 10) -> pd.DataFrame:
    """
    Recommend top-K similar papers for a selected seed paper.
    """
    if seed_index < 0 or seed_index >= len(df):
        raise IndexError("seed_index is out of range.")

    seed_vector = tfidf_matrix[seed_index]
    similarity_scores = cosine_similarity(seed_vector, tfidf_matrix).flatten()

    ranked_indices = similarity_scores.argsort()[::-1]
    ranked_indices = [idx for idx in ranked_indices if idx != seed_index]

    top_indices = ranked_indices[:k]

    results = df.iloc[top_indices].copy()
    results["similarity_score"] = similarity_scores[top_indices]

    return results[[
        "id",
        "title",
        "authors",
        "categories",
        "update_date",
        "similarity_score"
    ]]

In [ ]:
search_results = search_seed_paper(df, keyword="black hole", max_results=10)

search_results

In [ ]:
seed_index = search_results.index[0]
seed_paper = df.iloc[seed_index]

print("Seed paper:")
print("Title:", seed_paper["title"])
print("Categories:", seed_paper["categories"])
print("Authors:", seed_paper["authors"])

recommendations = recommend_similar_papers(
    seed_index=seed_index,
    df=df,
    tfidf_matrix=tfidf_matrix,
    k=10
)

recommendations

In [ ]:
print("Highest-related TF-IDF paper:")
print("Title:", recommendations.iloc[0]["title"])
print("Similarity:", recommendations.iloc[0]["similarity_score"])

In [ ]:
def category_precision_at_k(seed_categories: str, recommended_categories, k: int = 10) -> float:
    """
    Calculate Category Precision@K.
    A recommendation is counted as relevant if it shares at least one category
    with the seed paper.
    """
    seed_set = set(seed_categories.split())

    if len(seed_set) == 0:
        return 0.0

    recommended_categories = recommended_categories[:k]

    relevant_count = 0

    for cats in recommended_categories:
        rec_set = set(cats.split())
        if len(seed_set.intersection(rec_set)) > 0:
            relevant_count += 1

    return relevant_count / k

In [ ]:
import numpy as np
def recommend_similar_papers_sbert(seed_index, df, embeddings, k=10):
    seed_embedding = embeddings[seed_index]

    # Embeddings were normalized during encoding,
    # so the dot product is equivalent to cosine similarity.
    similarities = embeddings @ seed_embedding

    # Rank from highest to lowest similarity
    ranked_indices = np.argsort(similarities)[::-1]

    # Remove the seed paper itself
    ranked_indices = ranked_indices[ranked_indices != seed_index]

    # Select Top-K
    top_indices = ranked_indices[:k]

    results = df.loc[
        top_indices,
        ["id", "title", "authors", "categories", "update_date"]
    ].copy()

    results["similarity_score"] = similarities[top_indices]

    return results.reset_index(drop=True)

In [ ]:
sbert_recommendations = recommend_similar_papers_sbert(
    seed_index=seed_index,
    df=df,
    embeddings=sbert_embeddings,
    k=10
)

print("Seed paper:")
print("Title:", df.loc[seed_index, "title"])
print("Categories:", df.loc[seed_index, "categories"])

sbert_recommendations

In [ ]:
print("Highest-related SBERT paper:")
print("Title:", sbert_recommendations.iloc[0]["title"])
print("Similarity:", sbert_recommendations.iloc[0]["similarity_score"])

In [ ]:
precision = category_precision_at_k(
    seed_categories=seed_paper["categories"],
    recommended_categories=recommendations["categories"].tolist(),
    k=TOP_K
)

print(f"Category Precision@{TOP_K}: {precision:.2f}")

In [ ]:
def run_demo(keyword: str, result_number: int = 0, k: int = 10):
    """
    Search for a seed paper by keyword, select one result,
    generate top-K recommendations, and calculate Category Precision@K.
    """
    search_results = search_seed_paper(df, keyword=keyword, max_results=10)
    
    if len(search_results) == 0:
        print(f"No seed papers found for keyword: {keyword}")
        return None
    
    if result_number >= len(search_results):
        print(f"result_number is too large. Only {len(search_results)} results found.")
        return None
    
    seed_index = search_results.index[result_number]
    seed_paper = df.iloc[seed_index]
    
    print("=" * 80)
    print("Seed paper")
    print("=" * 80)
    print("ID:", seed_paper["id"])
    print("Title:", seed_paper["title"])
    print("Categories:", seed_paper["categories"])
    print("Authors:", seed_paper["authors"])
    print("Update date:", seed_paper["update_date"])
    
    recommendations = recommend_similar_papers(
        seed_index=seed_index,
        df=df,
        tfidf_matrix=tfidf_matrix,
        k=k
    )
    
    precision = category_precision_at_k(
        seed_categories=seed_paper["categories"],
        recommended_categories=recommendations["categories"].tolist(),
        k=k
    )
    
    print("\n" + "=" * 80)
    print(f"Top {k} recommendations")
    print("=" * 80)
    display(recommendations)
    
    print("\n" + "=" * 80)
    print(f"Category Precision@{k}: {precision:.2f}")
    print("=" * 80)
    
    return recommendations

In [ ]:
run_demo("black hole")

In [ ]:
search_seed_paper(df, keyword="cosmology", max_results=5)

In [ ]:
seed_index = 374

sbert_results = recommend_similar_papers_sbert(
    seed_index=seed_index,
    df=df,
    embeddings=sbert_embeddings,
    k=5
)

print("Seed:", df.loc[seed_index, "title"])
display(sbert_results[["title", "categories", "similarity_score"]])

In [ ]:
search_seed_paper(df, keyword="supernova", max_results=5)

In [ ]:
seed_index = 26

sbert_results = recommend_similar_papers_sbert(
    seed_index=seed_index,
    df=df,
    embeddings=sbert_embeddings,
    k=5
)

figure_results = sbert_results[
    ["title", "categories", "similarity_score"]
].copy()

figure_results["title"] = (
    figure_results["title"]
    .str.replace(r"\n", " ", regex=False)  # literal \n in metadata
    .str.replace(r"\s+", " ", regex=True)   # actual whitespace/newlines
    .str.strip()
)
print("Seed:", df.loc[seed_index, "title"])
pd.set_option("display.max_colwidth", None)
display(figure_results)

In [ ]:
def category_precision_at_k(seed_index, recommendations, df, k):
    seed_categories = set(df.loc[seed_index, "categories"].split())

    top_k = recommendations.head(k)

    relevant = 0
    for categories in top_k["categories"]:
        rec_categories = set(categories.split())

        if seed_categories.intersection(rec_categories):
            relevant += 1

    return relevant / k

In [ ]:
def evaluate_sbert_seed(seed_index):
    recommendations = recommend_similar_papers_sbert(
        seed_index=seed_index,
        df=df,
        embeddings=sbert_embeddings,
        k=20
    )

    return {
        "seed": df.loc[seed_index, "title"],
        "P@5": category_precision_at_k(
            seed_index, recommendations, df, 5
        ),
        "P@10": category_precision_at_k(
            seed_index, recommendations, df, 10
        ),
        "P@20": category_precision_at_k(
            seed_index, recommendations, df, 20
        )
    }

In [ ]:
black_hole_seed = 47
supernova_seed = 26
cosmo_seed=374

In [ ]:
results = [
    evaluate_sbert_seed(black_hole_seed),
    evaluate_sbert_seed(supernova_seed),
    evaluate_sbert_seed(cosmo_seed)
]

pd.DataFrame(results)

In [ ]:
def evaluate_both_models(seed_index):
    seed_categories = df.loc[seed_index, "categories"]

    rows = []
    for name, recs in [
        ("TF-IDF", recommend_similar_papers(seed_index, df, tfidf_matrix, k=20)),
        ("Sentence-BERT", recommend_similar_papers_sbert(seed_index, df, sbert_embeddings, k=20)),
    ]:
        rows.append({
            "seed": df.loc[seed_index, "title"],
            "model": name,
            **{
                f"P@{k}": sum(
                    bool(set(seed_categories.split()) & set(c.split()))
                    for c in recs["categories"].head(k)
                ) / k
                for k in (5, 10, 20)
            },
        })
    return rows

comparison = pd.DataFrame(
    row
    for seed_index in (47, 26, 374)
    for row in evaluate_both_models(seed_index)
)
display(comparison)